# Análise de impactos do teste de cupons

## Imports e Downloads dos dados

In [ ]:
import pandas as pd
import requests
import tarfile
import polars as pl
import os
import duckdb
from io import BytesIO
import matplotlib.pyplot as plt

In [ ]:
#===================
# Download dos dados
#===================

url_orders = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/order.json.gz" 
url_consumers = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/consumer.csv.gz" 
url_restaurants = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/restaurant.csv.gz" 
url_abtest = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/ab_test_ref.tar.gz" 

con = duckdb.connect()

# Teste A/B
response = requests.get(url_abtest, stream=True)

# with tarfile.open(fileobj=response.raw, mode="r:gz") as tar:
#     for member in tar.getmembers():
#         if member.isfile():
#             print(member.name)

# response = requests.get(url_abtest, stream=True)
# with tarfile.open("ab.tar.gz") as tar:
#     for member in tar:
#         if member.isfile() and not member.name.startswith(".") and member.name.endswith(".csv"):
#             print(member.name)
#             file = tar.extractfile(member)
#             df = pd.read_csv(file)


with tarfile.open(fileobj=BytesIO(response.raw.read()), mode="r:gz") as tar:
    for member in tar:
        if member.isfile() and not member.name.startswith(".") and member.name.endswith(".csv"):
            print(member.name)
            file = tar.extractfile(member)
            df_abtest = pd.read_csv(file)
#     for member in tar:
#         if member.name.endswith("ab_test_ref.csv"):
#             file = tar.extractfile(member)
#             df = pd.read_csv(file)
#             break

con.register("ab_test", df_abtest)

# Restaurantes
df_restaurants = pd.read_csv(url_restaurants, compression="gzip")
con.register("restaurants", df_restaurants)

# Consumers
df_customers = pd.read_csv(url_consumers,compression="gzip")
con.register("customers", df_customers)

# Orders
chunks = pd.read_json(
    url_orders,
    lines=True,
    chunksize=10000,  # diminui o chunk!
    compression="gzip"
)

orders_dir = "data/orders"

for i, chunk in enumerate(chunks):
    chunk.to_parquet(
        f"{orders_dir}/orders_part_{i}.parquet",
        engine="pyarrow",
        index=False
    )

con.execute("""
CREATE VIEW orders AS 
SELECT * FROM '../data/orders/*.parquet'
""")


In [ ]:
#===================
# Download dos dados
#===================

url_orders = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/order.json.gz" 
url_consumers = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/consumer.csv.gz" 
url_restaurants = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/restaurant.csv.gz" 
url_abtest = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/ab_test_ref.tar.gz" 

con = duckdb.connect()

# Teste A/B
response = requests.get(url_abtest, stream=True)

# with tarfile.open(fileobj=response.raw, mode="r:gz") as tar:
#     for member in tar.getmembers():
#         if member.isfile():
#             print(member.name)

# response = requests.get(url_abtest, stream=True)
# with tarfile.open("ab.tar.gz") as tar:
#     for member in tar:
#         if member.isfile() and not member.name.startswith(".") and member.name.endswith(".csv"):
#             print(member.name)
#             file = tar.extractfile(member)
#             df = pd.read_csv(file)


with tarfile.open(fileobj=BytesIO(response.raw.read()), mode="r:gz") as tar:
    for member in tar:
        if member.isfile() and not member.name.startswith(".") and member.name.endswith(".csv"):
            # print(member.name)
            file = tar.extractfile(member)
            df_abtest = pd.read_csv(file)
#     for member in tar:
#         if member.name.endswith("ab_test_ref.csv"):
#             file = tar.extractfile(member)
#             df = pd.read_csv(file)
#             break

con.register("ab_test", df_abtest)

# Restaurantes
df_restaurants = pd.read_csv(url_restaurants, compression="gzip")
con.register("restaurants", df_restaurants)

# Consumers
df_customers = pd.read_csv(url_consumers,compression="gzip")
con.register("customers", df_customers)

# # Orders
# chunks = pd.read_json(
#     url_orders,
#     lines=True,
#     chunksize=10000,  # diminui o chunk!
#     compression="gzip"
# )

# orders_dir = "data/orders"

# for i, chunk in enumerate(chunks):
#     chunk.to_parquet(
#         f"{orders_dir}/orders_part_{i}.parquet",
#         engine="pyarrow",
#         index=False
#     )

con.execute("""
CREATE VIEW orders AS 
SELECT * FROM '../data/orders/*.parquet'
""")


## Análise exploratória

In [ ]:
#===============================
# Usuários participando do teste
#===============================
df_customers_test = df_customers.merge(
    df_abtest,
    on='customer_id',
    how='left'
)

df_customers_test.groupby("is_target").agg(
    count_customers=("customer_id","count")
).reset_index()

,is_target,count_customers
0,control,360413
1,target,445743


In [17]:
df_customers_test.groupby(["is_target","active"]).agg(
    count_customers=("customer_id","count")
).reset_index()

,is_target,active,count_customers
0,control,False,713
1,control,True,359700
2,target,False,882
3,target,True,444861


In [ ]:
#===============================
# Usuários com 1 ou mais pedidos
#===============================
df_conv = con.execute("""
    WITH
    ORDERS_PER_USER as (
    SELECT 
        c.customer_id,
        COUNT(o.order_id) orders_amount
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE c.active
    GROUP BY 1
                
    )
    SELECT 
        a.is_target,
        case when coalesce(orders_amount,0) > 0 then 1 else 0 end as fl_order,
        COUNT(c.customer_id) as count_customers
    FROM customers c
    LEFT JOIN ab_test a ON c.customer_id = a.customer_id
    LEFT JOIN ORDERS_PER_USER o ON c.customer_id = o.customer_id
    WHERE c.active
    GROUP BY 1,2
""").fetchdf()

display(df_conv)


,is_target,fl_order,count_customers
0,control,1,359700
1,target,1,444861


In [ ]:
#====================
# VALORES DOS PEDIDOS
#====================
df_ab_orders = con.execute("""
    SELECT 
        a.is_target,
        COUNT(o.order_id) as orders_amount,
        COUNT(c.customer_id) as customers_amount,
        SUM(o.order_total_amount) / COUNT(c.customer_id) as order_amount_per_user,
        AVG(o.order_total_amount) / COUNT(c.customer_id) as avg_amount_per_user,
        COUNT(o.order_id) / COUNT(c.customer_id) as orders_per_user
    FROM customers c
    LEFT JOIN ab_test a ON c.customer_id = a.customer_id
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE c.active
    GROUP BY 1
""").fetchdf()

display(df_ab_orders)


,is_target,orders_amount,customers_amount,order_amount_per_user,avg_amount_per_user,orders_per_user
0,control,1521612,1521612,47.893628,0.000031,1.0
1,target,2130524,2130524,47.738018,0.000022,1.0


In [ ]:
#======================
# Atividade por usuário
#======================
df_order_user = con.execute("""
    with
    ORDERS AS (
        SELECT 
            a.is_target,
            COUNT(distinct c.customer_id) AS count_customers,
            COUNT(o.order_id) * 1.0 / COUNT(DISTINCT c.customer_id) AS orders_per_user
        FROM customers c
        LEFT JOIN ab_test a ON c.customer_id = a.customer_id
        LEFT JOIN orders o ON c.customer_id = o.customer_id 
        GROUP BY 1
    )
    SELECT 
        is_target,
        case 
            when orders_per_user  = 0 then '0 pedidos'
            when orders_per_user  = 1 then '1 pedido'
            when orders_per_user <= 5 then '5 pedidos'
            when orders_per_user <= 20 then '20 pedidos'
            when orders_per_user  <= 30 then '30 pedidos'
            when orders_per_user  <= 50 then '50 pedidos'
            when orders_per_user  > 50 then 'Mais 50 pedidos'
            else 'Sem info'
            end as orders_per_user_range,
        SUM(count_customers) AS customers_amount
    FROM ORDERS
    GROUP BY 1,2
# """).fetchdf()

len(df_order_user)

In [ ]:
# #===================
# # Análise de pedidos
# #===================
# df_ab_orders = con.execute("""
#     SELECT 
#         a.is_target,
#         o.customer_id,
#         o.merchant_id,
#         o.delivery_address_state,
#         o.order_scheduled,
#         o.origin_platform,
#         SUM(o.order_total_amount) as sum_orders_total_amount,
#         COUNT(o.order_id) as count_orders
#     FROM orders o
#     LEFT JOIN ab_test a ON o.customer_id = a.customer_id
#     LEFT JOIN customers c ON o.customer_id = c.customer_id
#     LEFT JOIN restaurants r ON o.merchant_id = r.id
#     WHERE c.active
#     GROUP BY 1,2,3,4,5,6
# """).fetchdf()

# len(df_ab_orders)


1737213

In [ ]:
#==================================
# Top restaurantes com mais pedidos
#==================================
df_ab_orders = con.execute("""
    SELECT 
        a.is_target,
        o.customer_id,
        o.merchant_id,
        COUNT(o.customer_id) as count_customers,                   
        COUNT(o.order_id) as count_orders
    FROM restaurants r
    LEFT JOIN orders o ON r.id = ON o.merchant_id
    LEFT JOIN ab_test a ON o.customer_id = a.customer_id
    LEFT JOIN customers c ON o.customer_id = c.customer_id
    WHERE c.active
    GROUP BY 1,2,3
    ORDER BY 5 DESC
    LIMIT 10
""").fetchdf()

len(df_ab_orders)


In [ ]:
#=================
# Tempo de entrega
#=================
df_delivery_time = con.execute("""
    SELECT 
        is_target,
        case
            when r.delivery_time < 1 then 'f. Sem informação'
            when r.delivery_time <= 15 then 'a. Até 10 minutos' 
            when r.delivery_time <= 30 then 'a. Até 30 minutos'
            when r.delivery_time <= 45 then 'b. 20-45 minutos'
            when r.delivery_time <= 60 then 'c. 45-60 minutos'
            when r.delivery_time <= 90 then 'd. 60-90 minutos'
            else 'f. Sem informação'
            end as delivery_time_range,
        count(r.id) as count_restaurants,
        count(o.order_id) as count_orders
    FROM restaurants r  
    LEFT JOIN orders o ON r.id = o.merchant_id
    LEFT JOIN ab_test a ON o.customer_id = a.customer_id
    group by 1,2 
    order by 1,2          
""").fetchdf()

print(df_delivery_time)

In [ ]:
df_aux = con.execute("""
    SELECT 
        is_target,
        enabled,
        price_range,
        COUNT(DISTINCT o.customer_id) AS customers_amount,
        COUNT(o.order_id) AS total_orders,
        COUNT(o.order_id) * 1.0 / COUNT(DISTINCT c.customer_id) AS orders_per_user,
        SUM(o.order_total_amount) * 1.0 / COUNT(DISTINCT c.customer_id) AS spent_per_user,
        
                     
    FROM customers c
    LEFT JOIN ab_test a ON c.customer_id = a.customer_id
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    LEFT JOIN restaurants r ON o.merchant_id = r.id
    WHERE active                
    GROUP BY 1,2,3
""").fetchdf()

print(df_aux)

In [ ]:
#===================
# Período de entrega
#===================

df_day_time = con.execute("""
    SELECT 
        a.is_target,
        case
            when EXTRACT(HOUR FROM order_created_at) < 5 THEN 'madrugada'
            when EXTRACT(HOUR FROM order_created_at) < 12 THEN 'manha'
            when EXTRACT(HOUR FROM order_created_at) < 18 THEN 'tarde'
            else 'noite'
            end as time_range,
        COUNT(distinct c.customer_id) AS customers_amount,
        COUNT(o.order_id) * 1.0 / COUNT(DISTINCT c.customer_id) AS orders_per_user,
        SUM(o.order_total_amount) * 1.0 / COUNT(DISTINCT c.customer_id) AS total_spent_per_user,
        AVG(o.order_total_amount) * 1.0 / COUNT(DISTINCT c.customer_id) AS avg_spent_per_user
    FROM customers c
    LEFT JOIN ab_test a ON c.customer_id = a.customer_id
    LEFT JOIN orders o ON c.customer_id = o.customer_id 
    LEFT JOIN restaurants r ON o.merchant_id = r.id
    GROUP BY 1,2
    ORDER BY 1,2
""").fetchdf()

display(df_day_time)

In [ ]:
#==============
# Dia da semana
#==============

df_week_day = con.execute("""
    SELECT 
        a.is_target,
        extract(dow from order_created_at) AS day_of_week,
        COUNT(distinct c.customer_id) AS customers_amount,
        COUNT(o.order_id) * 1.0 / COUNT(DISTINCT c.customer_id) AS orders_per_user,
        SUM(o.order_total_amount) * 1.0 / COUNT(DISTINCT c.customer_id) AS total_spent_per_user,
        AVG(o.order_total_amount) * 1.0 / COUNT(DISTINCT c.customer_id) AS avg_spent_per_user
    FROM customers c
    LEFT JOIN ab_test a ON c.customer_id = a.customer_id
    LEFT JOIN orders o ON c.customer_id = o.customer_id 
    LEFT JOIN restaurants r ON o.merchant_id = r.id
    GROUP BY 1,2
    ORDER BY 1,2
""").fetchdf()

dow_labels = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
dow_orders = df_week_day.groupby('day_of_week').size()
axes[2].bar(dow_orders.index, dow_orders.values, color='#4CAF50', alpha=0.8)
axes[2].set_xticks(range(7))
axes[2].set_xticklabels(dow_labels)
axes[2].set_title('Pedidos por Dia da Semana')
axes[2].set_ylabel('Nº de Pedidos')

plt.tight_layout()
plt.savefig('fig_01_distribuicao_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#===================
# Pedidos por estado
#===================



# Configuração global de visualização
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

df_aux = df_aux[df_aux["delivery_address_state"] != "AC"] # amostra muito pequena, pode sujar a análise

df_target = df_aux[df_aux["is_target"] == "target"]
df_control = df_aux[df_aux["is_target"] == "control"]

plt.figure()

plt.plot(
    df_target["delivery_address_state"],
    df_target["orders_per_user"]
    label = "Target"
)

plt.plot(
    df_control["delivery_address_state"],
    df_control["orders_per_user"],

    label = "Control"
)

plt.title("Controle vs Target por Estado")
plt.xlabel("Estado")
plt.ylabel("Order per user")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

plt.figure()

plt.plot(
    df_target["delivery_address_state"],
    df_target["total_spent_per_user"],
    label = "Target"
)

plt.plot(
    df_control["delivery_address_state"],
    df_control["total_spent_per_user"],
    label = "Control"
)

plt.title("Controle vs Target por Estado")
plt.xlabel("Estado")
plt.ylabel("Order per user")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()